In [5]:
import pandas as pd

df = pd.read_csv("../data/flight_price.csv")

In [6]:
X = df.drop("price", axis=1)
y = df["price"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX:")
print(X.head())

print("\ny:")
print(y.head())

X shape: (300153, 11)
y shape: (300153,)

X:
   Unnamed: 0   airline   flight source_city departure_time stops  \
0           0  SpiceJet  SG-8709       Delhi        Evening  zero   
1           1  SpiceJet  SG-8157       Delhi  Early_Morning  zero   
2           2   AirAsia   I5-764       Delhi  Early_Morning  zero   
3           3   Vistara   UK-995       Delhi        Morning  zero   
4           4   Vistara   UK-963       Delhi        Morning  zero   

    arrival_time destination_city    class  duration  days_left  
0          Night           Mumbai  Economy      2.17          1  
1        Morning           Mumbai  Economy      2.33          1  
2  Early_Morning           Mumbai  Economy      2.17          1  
3      Afternoon           Mumbai  Economy      2.25          1  
4        Morning           Mumbai  Economy      2.33          1  

y:
0    5953
1    5953
2    5956
3    5955
4    5955
Name: price, dtype: int64


In [51]:
X = df.drop(columns=["price", "Unnamed: 0"])
y = df["price"]


In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)



In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (240122, 11)
X_test: (60031, 11)
y_train: (240122,)
y_test: (60031,)


In [14]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(X_train[["class"]])

print(encoded[:5])
print(encoder.get_feature_names_out(["class"]))

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (5, 2)>
  Coords	Values
  (0, 1)	1.0
  (1, 1)	1.0
  (2, 0)	1.0
  (3, 1)	1.0
  (4, 0)	1.0
['class_Business' 'class_Economy']


In [9]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# numerical columns
numeric_features = ["duration", "days_left"]

# categorical columns
categorical_features = [
    "airline",
    "flight",
    "source_city",
    "departure_time",
    "stops",
    "arrival_time",
    "destination_city",
    "class"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("categ", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train:", X_train_processed.shape)
print("X_test:", X_test_processed.shape)

X_train: (240122, 1598)
X_test: (60031, 1598)


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ("preprocessor" , preprocessor),
    ("regressor" , LinearRegression())
   ])

model.fit(X_train , y_train)

y_pred = model.predict(X_test)

print(y_pred[:5])

[ 6350.40323987 56266.73218313 12732.23322418 53565.0473278
 11162.48129462]


In [13]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor",RandomForestRegressor(
        n_estimators = 2,
        random_state = 42,
        n_jobs =1
    ))
])

rf_model.fit(X_train,y_train)

y_pred_rf = rf_model.predict(X_test)

print(y_pred_rf[:5])

[ 6149.5 68807.   6195.  60160.   6578. ]


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score 
import numpy as np
# lineare reg
mae_lr = mean_absolute_error(y_test,y_pred)
rmse_lr = np.sqrt(mean_squared_error(y_test,y_pred))
r2_lr = r2_score(y_test,y_pred)

# random forest
mae_rf = mean_absolute_error(y_test,y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test,y_pred_rf)

print("Linear Regression")
print("MAE:", mae_lr)
print("RMSE:", rmse_lr)
print("R²:", r2_lr)

print("\nRandom Forest")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R²:", r2_rf)

Linear Regression
MAE: 4254.712530380953
RMSE: 6197.510292887257
R²: 0.9254889337619255

Random Forest
MAE: 946.3925283444822
RMSE: 2839.3392602427634
R²: 0.9843605729078891


In [15]:
y_train_pred_rf = rf_model.predict(X_train)

train_r2 = r2_score(y_train, y_train_pred_rf)
test_r2 = r2_score(y_test, y_pred_rf)

print("Train R²:", train_r2)
print("Test R²:", test_r2)

Train R²: 0.9956244810262114
Test R²: 0.9843605729078891


In [16]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring="r2",
    n_jobs=1
)


print("Scores R² :", cv_scores)
print("Moyenne R² :", cv_scores.mean())
print("Écart-type :", cv_scores.std())

Scores R² : [0.98420423 0.98373426 0.984849   0.98373306 0.98422255]
Moyenne R² : 0.9841486192877882
Écart-type : 0.0004107249936285596


In [17]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import numpy as np

# prediction , metriques finales

y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test,y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test,y_pred_rf))
r2_rf = r2_score(y_test,y_pred_rf)

print("MAE :", mae_rf)
print("RMSE :", rmse_rf)
print("R² :", r2_rf)

MAE : 946.3925283444822
RMSE : 2839.3392602427634
R² : 0.9843605729078891


In [18]:
absolute_errors = np.abs(y_test - y_pred_rf)

mae = absolute_errors.mean()
std = absolute_errors.std()
n = len(absolute_errors)

standard_error = std / np.sqrt(n)

ci_lower = mae - 1.96 * standard_error
ci_upper = mae + 1.96 * standard_error


print("MAE :", mae)
print("Écart-type :", std)
print("Erreur standard :", standard_error)
print("IC à 95 % :", (ci_lower, ci_upper))

MAE : 946.3925283444822
Écart-type : 2676.9960765451674
Erreur standard : 10.925968540891441
IC à 95 % : (np.float64(924.977630004335), np.float64(967.8074266846295))


In [19]:
import joblib

joblib.dump(rf_model,"random_forest_model.joblib")

['random_forest_model.joblib']